# Inference Evaluator — Gemma (local) vs MedGemma (HF API)

Evaluate **zero-shot** clinical outcome prediction on your test CSV using:
- Local: `google/gemma-3-270m` (runs on Apple Silicon **MPS**, CUDA **GPU**, or CPU automatically)
- Remote: `google/medgemma-27b-text-it` via Hugging Face **Inference API**

Metrics reported: **accuracy**, **macro-F1**, **latency** (mean & p95), and **resource usage** (CPU/RAM client-side).

> **Note (Nov 2025):** The Hugging Face Inference API moved to the Router endpoint. This notebook uses `https://router.huggingface.co/hf-inference/models/{model_id}`.


## 0) Setup
Install dependencies. For Apple Silicon, these CPU wheels enable MPS automatically on macOS ≥ 12.3.

In [8]:
!pip install --upgrade pip
!pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cpu
!pip install transformers scikit-learn psutil requests tqdm

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu


## 1) Configuration
Set your paths and options here. You can point `TEST_CSV` to any CSV that matches the expected schema.

In [9]:
import os

# === REQUIRED: Path to your test CSV ===
# Expected columns: patient_sex, patient_age, diagnosis, icd_code, treatment, comorbidity,
# systolic_bp, diastolic_bp, heart_rate, temperature, glucose, cholesterol, output_text
TEST_CSV = os.getenv("INFER_EVAL_TEST_CSV", "dataset/splits/test.csv")

# Labels to classify (must match values in output_text)
LABELS = ["Stable", "Improved", "Worsened"]

# Models
STUDENT_MODEL_ID = "google/gemma-3-270m-it"
TEACHER_MODEL_ID = "google/medgemma-4b-it"

# Hugging Face token for remote model
HF_TOKEN = os.getenv("HF_TOKEN" , "")  # set via: export HF_TOKEN=your_hf_token

# Inference parameters (deterministic)
GEN_KW = dict(max_new_tokens=2, do_sample=False)

# Optional: limit eval size for quick sanity checks
SAMPLE_LIMIT = None  # e.g., 100 or None for full test set

# Resource sampling interval (seconds)
STATS_INTERVAL = 0.25

print("Config loaded. TEST_CSV=", TEST_CSV)

Config loaded. TEST_CSV= dataset/splits/test.csv


## 2) Imports

In [10]:
import time, json, threading
import psutil
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import requests

## 3) Data Loading & Prompt Builder
Build a compact clinical vignette per row and a normalizer to map free-form outputs into the fixed label set.

In [11]:
df = pd.read_csv(TEST_CSV)
if SAMPLE_LIMIT is not None:
    df = df.sample(SAMPLE_LIMIT, random_state=0).reset_index(drop=True)

def row_to_prompt(r):
    return (
        f"Patient: {r['patient_sex']}, {int(r['patient_age'])} years old.\n"
        f"Diagnosis: {r['diagnosis']} (ICD-10: {r['icd_code']}).\n"
        f"Treatment: {r['treatment']}.\n"
        f"Comorbidity: {r['comorbidity']}.\n"
        f"Vitals — SBP: {r['systolic_bp']} mmHg, DBP: {r['diastolic_bp']} mmHg, "
        f"Heart Rate: {r['heart_rate']} bpm, Temp: {r['temperature']}°C.\n"
        f"Labs — Glucose: {r['glucose']} mg/dL, Cholesterol: {r['cholesterol']} mg/dL.\n\n"
        f"Question: Based on these findings, predict the clinical outcome.\n"
        f"Answer with one word ONLY from this set: {', '.join(LABELS)}."
    )

prompts = [row_to_prompt(r) for _, r in df.iterrows()]
gold = df["clinical_outcome"].tolist()

def normalize_label(s: str) -> str:
    s_low = s.lower()
    for lab in LABELS:
        if lab.lower() in s_low:
            return lab
    first = s.split()[0].strip(" .,:;").capitalize()
    return first if first in LABELS else "_OTHER_"

len(prompts), prompts[0][:200] + "..."

(2000,
 'Patient: Male, 88 years old.\nDiagnosis: Osteoarthritis (ICD-10: I25).\nTreatment: Dialysis.\nComorbidity: Obesity.\nVitals — SBP: 178 mmHg, DBP: 70 mmHg, Heart Rate: 84 bpm, Temp: 36.9°C.\nLabs — Glucose:...')

## 4) Resource Sampler (CPU/RAM)
Client-side CPU and RAM sampling while inference runs. GPU metrics are omitted for portability.

In [12]:
class ResourceSampler:
    def __init__(self, interval=0.25):
        self.interval = interval
        self._stop = threading.Event()
        self.samples = []

    def _sample_once(self):
        cpu = psutil.cpu_percent(interval=None)
        mem = psutil.virtual_memory().percent
        self.samples.append({"cpu_percent": cpu, "mem_percent": mem})

    def _run(self):
        while not self._stop.is_set():
            self._sample_once()
            time.sleep(self.interval)

    def start(self):
        self._stop.clear()
        self.thread = threading.Thread(target=self._run, daemon=True)
        self.thread.start()

    def stop(self):
        self._stop.set()
        self.thread.join()

    def summary(self):
        if not self.samples:
            return {}
        def max_of(key):
            return max(s.get(key, 0) for s in self.samples)
        def avg_of(key):
            vals = [s.get(key, 0) for s in self.samples]
            return sum(vals)/len(vals) if vals else 0.0
        return {
            "cpu_percent_max": round(max_of("cpu_percent"), 1),
            "cpu_percent_avg": round(avg_of("cpu_percent"), 1),
            "mem_percent_max": round(max_of("mem_percent"), 1),
            "mem_percent_avg": round(avg_of("mem_percent"), 1),
        }


## 5) Local Evaluation — Gemma-3-270M (MPS/CUDA/CPU)
Automatically uses **MPS** on Apple Silicon, **CUDA** if available, otherwise **CPU**.

In [13]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

def pick_device():
    if torch.backends.mps.is_available():
        print("⚡ Using Apple Silicon (MPS) acceleration.")
        return torch.device("mps"), torch.float32
    if torch.cuda.is_available():
        print("🟢 Using CUDA GPU.")
        return torch.device("cuda"), torch.float16
    print("🧠 Using CPU.")
    return torch.device("cpu"), torch.float32

def eval_local(model_id, prompts, gold):
    device, dtype = pick_device()
    model = AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype)
    tok = AutoTokenizer.from_pretrained(model_id)
    model.to(device)

    # Chat-style generator for Gemma-IT
    def generate_chat(p: str):
        messages = [
            {"role": "system", "content": "You are a medical classifier. Reply with exactly one word from: Stable, Improved, Worsened."},
            {"role": "user", "content": p}
        ]
        chat_text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(chat_text, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=16,
                do_sample=False,
                eos_token_id=tok.eos_token_id,
                pad_token_id=tok.eos_token_id,
                use_cache=True,
            )
        gen_ids = out[0][inputs["input_ids"].shape[1]:]
        gen_text = tok.decode(gen_ids, skip_special_tokens=True).strip()
        return gen_text

    # resource sampler
    sampler = ResourceSampler(interval=STATS_INTERVAL)
    sampler.start()
    latencies, preds = [], []

    t0 = time.time()
    for p in tqdm(prompts, desc=f"Local eval: {model_id}"):
        s = time.time()
        gen_text = generate_chat(p)
        latencies.append(time.time() - s)
        preds.append(normalize_label(gen_text))

    total = time.time() - t0
    sampler.stop()
    stats = sampler.summary()

    # sanity-check lengths
    if len(preds) != len(gold):
        raise ValueError(f"Pred/gold mismatch: {len(preds)} vs {len(gold)}")

    acc = accuracy_score(gold, preds)
    f1m = f1_score(gold, preds, average="macro", zero_division=0)
    report = classification_report(gold, preds, labels=LABELS, zero_division=0)

    print(f"✅ Completed local inference for {model_id}")

    return {
        "model_id": model_id,
        "accuracy": acc,
        "macro_f1": f1m,
        "per_sample_latency_sec_mean": sum(latencies)/len(latencies),
        "per_sample_latency_sec_p95": sorted(latencies)[int(0.95*len(latencies))-1],
        "total_time_sec": total,
        "resource_stats": stats,
        "report": report,
    }



## 7) Run & Compare
Generates a side-by-side comparison and saves a JSON report for later analysis.

In [14]:
results = {}

results["gemma3_270m"] = eval_local(STUDENT_MODEL_ID, prompts, gold)
# results["medgemma_4b_it"] = eval_local(TEACHER_MODEL_ID, prompts, gold)

def fmt_pct(x): return f"{100*x:.2f}%"
def fmt_sec(x): return f"{x:.3f}s"

def summarize(r):
    rs = r.get("resource_stats", {})
    return {
        "model": r["model_id"],
        "accuracy": fmt_pct(r["accuracy"]),
        "macro_f1": fmt_pct(r["macro_f1"]),
        "lat_mean": fmt_sec(r["per_sample_latency_sec_mean"]),
        "lat_p95": fmt_sec(r["per_sample_latency_sec_p95"]),
        "total_time": fmt_sec(r["total_time_sec"]),
        "cpu_max%": rs.get("cpu_percent_max"),
        "ram_max%": rs.get("mem_percent_max"),
    }

summary_df = pd.DataFrame([summarize(results[k]) for k in results])
display(summary_df)

os.makedirs("dataset/eval_reports", exist_ok=True)

# with open("dataset/eval_reports/inference_eval_summary_medgemma.json", "w") as f:
#     json.dump(results, f, indent=2)
# print('Saved detailed JSON to dataset/eval_reports/inference_eval_summary_medgemma.json')
# print(results["medgemma_4b_it"]["report"])

with open("dataset/eval_reports/inference_eval_summary.json", "w") as f:
    json.dump(results, f, indent=2)
print('Saved detailed JSON to dataset/eval_reports/inference_eval_summary.json')
print(results["gemma3_270m"]["report"])


⚡ Using Apple Silicon (MPS) acceleration.


Local eval: google/gemma-3-270m-it: 100%|██████████| 2000/2000 [11:56<00:00,  2.79it/s]


✅ Completed local inference for google/gemma-3-270m-it


,model,accuracy,macro_f1,lat_mean,lat_p95,total_time,cpu_max%,ram_max%
0,google/gemma-3-270m-it,33.40%,19.81%,0.358s,0.378s,716.513s,77.0,85.4


Saved detailed JSON to dataset/eval_reports/inference_eval_summary.json
              precision    recall  f1-score   support

      Stable       0.37      0.06      0.10       668
    Improved       0.33      0.94      0.49       665
    Worsened       0.00      0.00      0.00       667

    accuracy                           0.33      2000
   macro avg       0.23      0.33      0.20      2000
weighted avg       0.23      0.33      0.20      2000



## 8) Tips
- Override `TEST_CSV` by setting env var `INFER_EVAL_TEST_CSV` or editing the config cell.
- Use `SAMPLE_LIMIT=50` for quick trials.
- If you don't have HF API access to the MedGemma model, you can skip remote eval by commenting that call.
- The local device is auto-selected: MPS (Apple Silicon) → CUDA → CPU.